# NumPy — Vectorised Computation
**Assignment 1: Machine Learning Data Foundations**

*Condition: No Python `for` loops over array elements anywhere in this assignment.*


In [1]:
import numpy as np
import pandas as pd
import time
from numpy.lib.stride_tricks import sliding_window_view

# Ensure reproducibility
np.random.seed(42)
print("NumPy seed set to 42")


NumPy seed set to 42


### 1. Normal Distribution Array Creation and Memory Size Conversion
Create an array `A` of shape `(1000, 5)` from a normal distribution with mean 50 and standard deviation 12. Print its shape, dtype, and memory size in bytes. Convert it to `float32` and print the memory size again.

In [2]:
# Create normal distribution array (mean=50, std=12)
A = np.random.normal(loc=50.0, scale=12.0, size=(1000, 5))
print(f"Original A shape: {A.shape}")
print(f"Original A dtype: {A.dtype}")
print(f"Original A memory size: {A.nbytes} bytes")

# Convert to float32 and inspect memory
A_float32 = A.astype(np.float32)
print(f"Float32 A dtype: {A_float32.dtype}")
print(f"Float32 A memory size: {A_float32.nbytes} bytes (50% reduction)")


Original A shape: (1000, 5)
Original A dtype: float64
Original A memory size: 40000 bytes
Float32 A dtype: float32
Float32 A memory size: 20000 bytes (50% reduction)


### 2. Column Standardisation via Broadcasting
Standardise every column so each has mean 0 and standard deviation 1 in one expression using broadcasting. Print the column means and standard deviations after scaling, rounded to 6 decimals.

In [3]:
# Standardise in one expression using broadcasting
A = (A - A.mean(axis=0)) / A.std(axis=0)

col_means = np.round(A.mean(axis=0), 6)
col_stds = np.round(A.std(axis=0), 6)

print("Standardised Column Means:", col_means)
print("Standardised Column Stds: ", col_stds)


Standardised Column Means: [-0. -0.  0. -0. -0.]
Standardised Column Stds:  [1. 1. 1. 1. 1.]


### 3. Axis-Generic Min-Max Scaling Function
Write `minmax(X, axis)` that scales to the range 0 to 1 along any axis. Test it on `axis=0` and `axis=1` and show that the minimum and maximum come out as 0 and 1.

In [4]:
def minmax(X, axis):
    """Scales array X to range [0, 1] along the specified axis using broadcasting."""
    x_min = X.min(axis=axis, keepdims=True)
    x_max = X.max(axis=axis, keepdims=True)
    return (X - x_min) / (x_max - x_min)

# Test on axis=0 (column-wise scaling)
scaled_ax0 = minmax(A, axis=0)
print("axis=0 min per column:", np.round(scaled_ax0.min(axis=0), 6))
print("axis=0 max per column:", np.round(scaled_ax0.max(axis=0), 6))

# Test on axis=1 (row-wise scaling)
scaled_ax1 = minmax(A, axis=1)
print("axis=1 min across rows (min, max):", scaled_ax1.min(axis=1).min(), scaled_ax1.min(axis=1).max())
print("axis=1 max across rows (min, max):", scaled_ax1.max(axis=1).min(), scaled_ax1.max(axis=1).max())


axis=0 min per column: [0. 0. 0. 0. 0.]
axis=0 max per column: [1. 1. 1. 1. 1.]
axis=1 min across rows (min, max): 0.0 0.0
axis=1 max across rows (min, max): 1.0 1.0


### 4. Random NaN Injection and Column-Mean Imputation
Inject NaN into 5% of the positions of `A` at random. Replace every NaN with the mean of its own column, using masking and `np.nanmean`. Confirm with `np.isnan(A).sum() == 0`.

In [5]:
# Randomly inject NaNs into 5% of elements
nan_mask = np.random.choice([True, False], size=A.shape, p=[0.05, 0.95])
A[nan_mask] = np.nan
print(f"Total NaNs injected: {np.isnan(A).sum()}")

# Compute column-wise means ignoring NaNs
col_nan_means = np.nanmean(A, axis=0)

# Replace NaNs with column means using broadcasting and masking
broadcast_means = np.broadcast_to(col_nan_means, A.shape)
A = np.where(np.isnan(A), broadcast_means, A)

print("Verification (isnan sum == 0):", np.isnan(A).sum() == 0)


Total NaNs injected: 262
Verification (isnan sum == 0): True


### 5. Outlier Detection and Replacement with Column Median
Flag every value more than 3 standard deviations from its column mean and replace it with the column median. Report how many values were replaced per column.

In [6]:
means = A.mean(axis=0)
stds = A.std(axis=0)
medians = np.median(A, axis=0)

# Identify outliers: abs(value - mean) > 3 * std
outlier_mask = np.abs(A - means) > 3 * stds
replaced_per_col = outlier_mask.sum(axis=0)

# Replace outliers with column medians
broadcast_medians = np.broadcast_to(medians, A.shape)
A = np.where(outlier_mask, broadcast_medians, A)

print("Number of outlier values replaced per column:", replaced_per_col)
print("Total outliers replaced across array:", replaced_per_col.sum())


Number of outlier values replaced per column: [5 1 5 3 5]
Total outliers replaced across array: 19


### 6. Vectorized Euclidean Distance Matrix
Take the first 200 rows and build the full 200 x 200 Euclidean distance matrix using broadcasting only. Do not use loops, scipy, or sklearn. Prove it is correct by checking that the diagonal is 0 and that the matrix equals its own transpose.

In [7]:
# Extract first 200 rows
sub_A = A[:200, :]  # Shape (200, 5)

# Pairwise differences via broadcasting: (200, 1, 5) - (1, 200, 5) -> (200, 200, 5)
diffs = sub_A[:, np.newaxis, :] - sub_A[np.newaxis, :, :]
# Euclidean distance: sqrt(sum(diffs**2, axis=-1))
dist_matrix = np.sqrt(np.sum(diffs ** 2, axis=-1))

print(f"Distance matrix shape: {dist_matrix.shape}")

# Verification: diagonal is zero and symmetric matrix equals its transpose
diag_zero = np.allclose(np.diag(dist_matrix), 0.0)
is_symmetric = np.allclose(dist_matrix, dist_matrix.T)
print(f"Diagonal is all zeros: {diag_zero}")
print(f"Distance matrix equals its transpose: {is_symmetric}")


Distance matrix shape: (200, 200)
Diagonal is all zeros: True
Distance matrix equals its transpose: True


### 7. Find 5 Nearest Neighbours using `argpartition`
From that distance matrix, find for every row the indices of its 5 nearest rows, excluding itself. Return the result as a (200, 5) integer array. Use `argpartition`, not a sort of the whole row, and say in one line why.

In [8]:
# Distance to self is 0 on diagonal. By partitioning elements 1 through 5,
# we isolate the 5 smallest positive distances without a full O(N log N) sort.
partitioned_indices = np.argpartition(dist_matrix, kth=range(1, 6), axis=1)[:, 1:6]

print(f"Nearest indices array shape: {partitioned_indices.shape}")
print(f"Dtype: {partitioned_indices.dtype}")
print("First 3 rows of 5-nearest neighbor indices:\n", partitioned_indices[:3])


Nearest indices array shape: (200, 5)
Dtype: int64
First 3 rows of 5-nearest neighbor indices:
 [[180 179 178 116 182]
 [199 114  17  70  31]
 [ 52 146 113 132  38]]


*Why `argpartition`*: `np.argpartition` runs in linear time $O(N)$ to locate the top-k smallest distances without incurring the redundant $O(N \log N)$ cost of sorting all 200 items in each row.

### 8. Performance Benchmark: `np.bincount` vs Plain Python Loop
Create a 1-D array of 1,000,000 random integers between 0 and 99. Count how many times each value appears using `np.bincount`. Then count the same thing with a plain Python loop. Put `%%time` at the top of each cell and report the speed-up as a number.

In [9]:
rand_arr = np.random.randint(0, 100, size=1_000_000)
print(f"Array length: {len(rand_arr):,}")


Array length: 1,000,000


In [10]:
# NumPy vectorised bincount
t0 = time.perf_counter()
counts_numpy = np.bincount(rand_arr, minlength=100)
t1 = time.perf_counter()
np_time = t1 - t0
print(f"NumPy bincount time: {np_time * 1000:.3f} ms")


NumPy bincount time: 5.784 ms


In [11]:
# Plain Python loop
t0 = time.perf_counter()
counts_py = [0] * 100
for val in rand_arr:
    counts_py[val] += 1
t1 = time.perf_counter()
py_time = t1 - t0
print(f"Python loop time: {py_time * 1000:.3f} ms")

# Verify equality and calculate speedup
assert np.array_equal(counts_numpy, counts_py), "Counts do not match!"
speedup = py_time / np_time
print(f"Speed-up factor: {speedup:.2f}x faster with NumPy!")


Python loop time: 207.072 ms
Speed-up factor: 35.80x faster with NumPy!


### 9. Sliding Window Operations via `sliding_window_view`
Build all sliding windows of length 10 over a 1-D array of 10,000 values using `sliding_window_view`. Print the resulting shape. Compute the mean of each window and compare your answer against `pandas.Series.rolling(10).mean()` with `np.allclose`.

In [12]:
series_data = np.random.randn(10000)

# Build sliding windows of length 10
windows = sliding_window_view(series_data, window_shape=10)
print(f"Sliding windows shape: {windows.shape}")

# Compute window means using numpy
numpy_window_means = windows.mean(axis=1)

# Compute rolling means using pandas
pandas_window_means = pd.Series(series_data).rolling(10).mean().dropna().to_numpy()

# Compare
match = np.allclose(numpy_window_means, pandas_window_means)
print(f"NumPy and Pandas rolling means match perfectly: {match}")


Sliding windows shape: (9991, 10)
NumPy and Pandas rolling means match perfectly: True


### 10. Multi-Dimensional Reshaping and Axis Reduction
Reshape a `(12, 8)` array into `(4, 3, 8)`, then take the mean over the middle axis. Explain in two lines what the resulting shape represents.

In [13]:
arr_12_8 = np.arange(96).reshape(12, 8)
reshaped = arr_12_8.reshape(4, 3, 8)
reduced_mean = reshaped.mean(axis=1)

print(f"Original shape: {arr_12_8.shape}")
print(f"Reshaped shape: {reshaped.shape}")
print(f"Resulting mean shape: {reduced_mean.shape}")


Original shape: (12, 8)
Reshaped shape: (4, 3, 8)
Resulting mean shape: (4, 8)


*Explanation of resulting shape (4, 8)*:
The `(4, 8)` shape represents the aggregation of the original 12 rows into 4 distinct groups, where each of the 4 output rows is the element-wise average of 3 consecutive chunk rows across all 8 feature dimensions.

### 11. Serialization and Verification with `.npy`
Save your cleaned array with `np.save("clean.npy", A)`, load it back with `np.load`, and assert that the loaded array is identical to the original.

In [14]:
# Save array to disk
np.save("clean.npy", A)

# Load array back
A_loaded = np.load("clean.npy")

# Verify equality
assert np.array_equal(A, A_loaded), "Loaded array is not identical!"
print("Verification Success: Loaded array is 100% identical to original clean array.")
print(f"Clean array shape: {A_loaded.shape}, dtype: {A_loaded.dtype}")


Verification Success: Loaded array is 100% identical to original clean array.
Clean array shape: (1000, 5), dtype: float64
